**Views vs Copies Practices**

In [1]:
import numpy as np

| مفهوم                          | **View**                                          | **Copy**                              |
| ------------------------------ | ------------------------------------------------- | ------------------------------------- |
| داده‌ی جدید می‌سازد؟           | ❌ معمولاً نه                                      | ✅ بله                                 |
| حافظه‌ی جداگانه دارد؟          | ❌ معمولاً همان حافظه‌ی آرایه اصلی                 | ✅ حافظه‌ی مستقل                       |
| تغییر روی آرایه اصلی اثر دارد؟ | ✅ معمولاً بله                                     | ❌ خیر                                 |
| ساخت آن                        | سریع‌تر و کم‌مصرف‌تر                              | کندتر و پرمصرف‌تر                     |
| مثال                           | `view = arr[1:4]`                                 | `copy = arr[1:4].copy()`              |
| کاربرد                         | وقتی فقط می‌خواهی به داده موجود دسترسی داشته باشی | وقتی می‌خواهی داده را مستقل تغییر دهی |

```text
view.base is arr   # True اگه View باشه (به آرایه‌ی اصلی اشاره داره)
copy.base is None   # True اگه Copy مستقل باشه

--------

##### <mark>Exercise 1 — Detecting View vs Copy</mark>
```text
Given the following operations, predict (before running) whether each result is a View or a Copy, then verify using .base:

arr = np.array([10, 20, 30, 40, 50])

a = arr[1:3]
b = arr[[0, 2, 4]]
c = arr[arr > 20]
d = arr.reshape(5, 1)

For each of a, b, c, d, print whether it's a View or Copy, and confirm by modifying it and checking if arr changed.

In [2]:
# -------------------------
# a: Slicing
# -------------------------
arr = np.array([10, 20, 30, 40, 50])

a = arr[1:3]

print("a:", "View" if a.base is not None else "Copy")
a[0] = 999

print("arr after modifying a:", arr)

# -------------------------
# b: Fancy Indexing
# -------------------------
arr = np.array([10, 20, 30, 40, 50])

b = arr[[0, 2, 4]]

print("b:", "View" if b.base is not None else "Copy")
b[0] = 999

print("arr after modifying b:", arr)

# -------------------------
# c: Boolean Indexing
# -------------------------
arr = np.array([10, 20, 30, 40, 50])

c = arr[arr > 20]

print("c:", "View" if c.base is not None else "Copy")
c[0] = 999

print("arr after modifying c:", arr)

# -------------------------
# d: Reshape
# -------------------------
arr = np.array([10, 20, 30, 40, 50])

d = arr.reshape(5, 1)

print("d:", "View" if d.base is not None else "Copy")
d[0, 0] = 999

print("arr after modifying d:", arr)

a: View
arr after modifying a: [ 10 999  30  40  50]
b: Copy
arr after modifying b: [10 20 30 40 50]
c: Copy
arr after modifying c: [10 20 30 40 50]
d: View
arr after modifying d: [999  20  30  40  50]


------

##### <mark>Exercise 2 — The Silent Bug</mark>
```text
You're preprocessing a batch of data for a model. Write code that reproduces this common bug:

dataset = np.array([100, 200, 300, 400, 500])
batch = dataset[0:3]     # intended to be an independent working copy
batch[:] = 0             # "clearing" the batch for reuse

Show that this unintentionally corrupts dataset. 
Then fix the code using .copy() so that modifying batch no longer affects dataset.

In [9]:
#🔴 Bug — بدون .copy()
dataset = np.array([100, 200, 300, 400, 500])

batch = dataset[0:3]      # View
batch[:] = 0              # Modifies the shared data

print("batch:", batch)
print("dataset:", dataset)

print("-------------------------------")
#🟢 Fix — استفاده از .copy()
dataset = np.array([100, 200, 300, 400, 500])

batch = dataset[0:3].copy()    # Independent copy

batch[:] = 0

print("batch:", batch)
print("dataset:", dataset)

batch: [0 0 0]
dataset: [  0   0   0 400 500]
-------------------------------
batch: [0 0 0]
dataset: [100 200 300 400 500]


-----------

##### <mark>Exercise 3 — Memory-Efficient Pipeline</mark>
```text
You have a large array simulating a big dataset:

big_data = np.arange(1_000_000)

Write a small pipeline that: 
(1) takes a slice of the first 100,000 elements as a View (no unnecessary copying) to inspect them, 
(2) then creates an explicit, independent copy of just the elements greater than 500,000 
    for a separate "outliers" analysis that shouldn't affect the original array. 
    Confirm memory independence by modifying the outliers copy and checking big_data is unaffected.

In [14]:
big_data = np.arange(1_000_000)

view_batch = big_data[0:100_000]

copy_batch = big_data[big_data > 500_000].copy()

print("big_data size:", big_data.size)
print("view_batch size:", view_batch.size)
print("copy_batch size:", copy_batch.size)

# Check that view_batch shares data with big_data
print("view_batch is a view:", view_batch.base is not None)

# Check that copy_batch is independent
print("copy_batch is independent:", copy_batch.base is None)

# Modify the copy
copy_batch[:] = 0

# big_data should remain unchanged
print("big_data[500001]:", big_data[500001])
print("copy_batch[0]:", copy_batch[0])

big_data size: 1000000
view_batch size: 100000
copy_batch size: 499999
view_batch is a view: True
copy_batch is independent: True
big_data[500001]: 500001
copy_batch[0]: 0
